# 03_training — V1.1

Private input version1, frozen parent Pilot Kaggle145002634 v2, run `k230-topface-v11-pilot-1000-a02`, Drive archive SHA `f559a9df0fe01ea17884ce63671e6e89a162a037faa0a80045eec574e417430f`. The cloud worker reads that immutable archive from the mounted Kaggle notebook output, verifies all bytes and archive members, then runs this phase. Raw data and checkpoints stay in cloud; host collects bounded receipts only.

The API starts this private job with Internet disabled. The worker has 9000 seconds to verify inputs, run the experiment and save the output. The authorized GitHub Actions publisher uploads the saved archive to Drive and verifies it by reading back every byte.

Training uses CUDA0, FP32, no dependency installs and no large hue shifts. Logical sequence: 16 train images overfit (max1000updates), fresh pretrained20epoch Pilot, fresh random-initialization20epoch controlled comparison. Overfit failure stops before Pilot training. The two Pilot runs use identical splits, augmentation seeds, hyperparameters and epoch caps. Only val selects best; no synthetic_test or real test evaluation. Save best/last/optimizer/RNG/TensorBoard/config/metrics/SHA; failed computations are archived and never automatically replayed.


In [ ]:
"""Locate one hash-bound input across Kaggle's supported mount layouts."""
import hashlib
from pathlib import Path


def locate_input(root, manifest_sha256):
    root = Path(root).resolve()
    matches = []
    candidates = list(root.rglob("INPUT_MANIFEST.json"))
    if len(candidates) > 20:
        raise RuntimeError("Too many candidate manifests; require explicit input review")
    for candidate in candidates:
        candidate = candidate.resolve()
        if not candidate.is_relative_to(root):
            raise ValueError("Input manifest resolves outside Kaggle input mount")
        if hashlib.sha256(candidate.read_bytes()).hexdigest() == manifest_sha256:
            matches.append(candidate.parent)
    if len(matches) != 1:
        names = [str(path.relative_to(root)) for path in candidates]
        raise RuntimeError(f"Expected exactly one hash-bound input, found {len(matches)}; candidates={names}")
    return matches[0]

import sys
input_root=locate_input('/kaggle/input','cab8469436412a9efa1271cfa169f006f489c88edd92911aa9e346cb6f561e02')
sys.path.insert(0,str(input_root))
from downstream_job import main
main(input_root,'k230-topface-v11-training-a01','training',max_seconds=9000)
